In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class GarminDataValidator:
    """Garmin 資料品質驗證器"""

    def __init__(self):
        self.validation_results = {}

    def validate_sleep_times(self, sleeps_df, sleeps_levels_df):
        """驗證睡眠時間邏輯"""
        print("=== 睡眠時間邏輯驗證 ===\n")

        # 1. 轉換時間戳記為可讀時間
        sleeps_df['start_datetime'] = pd.to_datetime(sleeps_df['startTimeInSeconds'], unit='s')
        sleeps_df['sleep_duration_hours'] = sleeps_df['durationInSeconds'] / 3600
        sleeps_df['end_datetime'] = sleeps_df['start_datetime'] + pd.to_timedelta(sleeps_df['durationInSeconds'], unit='s')

        # 2. 檢查睡眠時長合理性
        print("睡眠時長統計：")
        print(f"最短睡眠: {sleeps_df['sleep_duration_hours'].min():.2f} 小時")
        print(f"最長睡眠: {sleeps_df['sleep_duration_hours'].max():.2f} 小時")
        print(f"平均睡眠: {sleeps_df['sleep_duration_hours'].mean():.2f} 小時")
        print(f"標準差: {sleeps_df['sleep_duration_hours'].std():.2f} 小時\n")

        # 3. 找出異常睡眠時長
        abnormal_duration = sleeps_df[(sleeps_df['sleep_duration_hours'] < 2) |
                                     (sleeps_df['sleep_duration_hours'] > 14)]
        if len(abnormal_duration) > 0:
            print(f"發現 {len(abnormal_duration)} 筆異常睡眠時長記錄")
            print("範例：")
            print(abnormal_duration[['_userId', 'date', 'sleep_duration_hours',
                                    'start_datetime', 'end_datetime']].head())

        # 4. 檢查日期歸屬邏輯
        print("\n=== 日期歸屬邏輯檢查 ===")
        sleeps_df['recorded_date'] = pd.to_datetime(sleeps_df['date'])
        sleeps_df['start_date'] = sleeps_df['start_datetime'].dt.date
        sleeps_df['end_date'] = sleeps_df['end_datetime'].dt.date

        # 分析日期不一致的情況
        date_mismatch = sleeps_df[sleeps_df['recorded_date'].dt.date != sleeps_df['start_date']]
        if len(date_mismatch) > 0:
            print(f"\n發現 {len(date_mismatch)} 筆記錄的 date 欄位與實際開始日期不符")

            # 分析不一致的模式
            sleeps_df['start_hour'] = sleeps_df['start_datetime'].dt.hour
            mismatch_analysis = date_mismatch.groupby('start_hour').size()
            print("\n依開始小時分組的不一致記錄數：")
            print(mismatch_analysis.sort_index())

        return sleeps_df

    def validate_sleep_stages(self, sleeps_df, levels_df):
        """驗證睡眠階段一致性"""
        print("\n=== 睡眠階段一致性驗證 ===\n")

        # 1. 檢查每個 summaryId 的階段連續性
        consistency_issues = []

        for summary_id in levels_df['summaryId'].unique()[:10]:  # 先檢查前10個
            stages = levels_df[levels_df['summaryId'] == summary_id].sort_values('startTimeInSeconds')

            # 檢查時間連續性
            for i in range(len(stages) - 1):
                current_end = stages.iloc[i]['endTimeInSeconds']
                next_start = stages.iloc[i + 1]['startTimeInSeconds']

                if current_end != next_start:
                    gap = next_start - current_end
                    consistency_issues.append({
                        'summaryId': summary_id,
                        'gap_seconds': gap,
                        'between_stages': f"{stages.iloc[i]['sleepLevels']} -> {stages.iloc[i+1]['sleepLevels']}"
                    })

        if consistency_issues:
            print(f"發現 {len(consistency_issues)} 個時間不連續問題")
            issues_df = pd.DataFrame(consistency_issues)
            print("\n時間間隙統計：")
            print(issues_df['gap_seconds'].describe())

        # 2. 驗證總時長匹配
        print("\n=== 總時長匹配驗證 ===")

        # 計算 levels 中每個 summary 的總時長
        levels_duration = levels_df.groupby('summaryId').apply(
            lambda x: (x['endTimeInSeconds'] - x['startTimeInSeconds']).sum()
        ).reset_index(columns=['calculated_duration'])

        # 與 sleeps 表對比
        duration_check = pd.merge(
            sleeps_df[['summaryId', 'durationInSeconds']],
            levels_duration,
            on='summaryId',
            how='inner'
        )

        duration_check['duration_diff'] = abs(
            duration_check['durationInSeconds'] - duration_check['calculated_duration']
        )

        mismatches = duration_check[duration_check['duration_diff'] > 60]  # 超過1分鐘差異
        if len(mismatches) > 0:
            print(f"發現 {len(mismatches)} 筆總時長不匹配的記錄")
            print(f"平均差異: {mismatches['duration_diff'].mean():.2f} 秒")

        return consistency_issues

    def validate_daily_activity(self, dailies_df):
        """驗證日活動資料邏輯"""
        print("\n=== 日活動資料驗證 ===\n")

        # 1. 檢查活動時間邏輯
        dailies_df['total_activity_time'] = (dailies_df['moderateIntensityDurationInSecon'] +
                                            dailies_df['vigorousIntensityDurationInSecon'])

        time_logic_issues = dailies_df[
            dailies_df['total_activity_time'] > dailies_df['activeTimeInSeconds']
        ]

        if len(time_logic_issues) > 0:
            print(f"發現 {len(time_logic_issues)} 筆活動時間邏輯錯誤")

        # 2. 檢查熱量邏輯
        dailies_df['calculated_active_cal'] = (dailies_df['bmrKilocalories'] +
                                              dailies_df['activeKilocalories'])

        # 3. 檢查步數合理性
        print("\n步數統計：")
        print(f"最高步數: {dailies_df['steps'].max():,}")
        print(f"平均步數: {dailies_df['steps'].mean():,.0f}")

        extreme_steps = dailies_df[dailies_df['steps'] > 40000]
        if len(extreme_steps) > 0:
            print(f"\n發現 {len(extreme_steps)} 天步數超過 40,000")

        return dailies_df

    def cross_table_validation(self, sleeps_df, dailies_df, epochs_df):
        """跨表格驗證"""
        print("\n=== 跨表格一致性驗證 ===\n")

        # 1. 檢查睡眠與日活動的日期對應
        sleep_dates = set(sleeps_df[['_userId', 'date']].apply(tuple, axis=1))
        daily_dates = set(dailies_df[['_userId', 'date']].apply(tuple, axis=1))

        only_in_sleep = sleep_dates - daily_dates
        only_in_daily = daily_dates - sleep_dates

        print(f"只在睡眠資料中的記錄: {len(only_in_sleep)}")
        print(f"只在日活動資料中的記錄: {len(only_in_daily)}")

        # 2. 檢查時間重疊（如果有 epochs 資料）
        if epochs_df is not None and len(epochs_df) > 0:
            # 這裡可以加入更詳細的時間重疊檢查
            pass

        return {
            'only_sleep': only_in_sleep,
            'only_daily': only_in_daily
        }

# 使用範例
def main():
    """主要執行函數"""
    # 讀取資料
    print("讀取資料中...")

    # 這裡需要替換為實際的檔案路徑
    try:
        sleeps_df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleeps.csv')
        sleeps_levels_df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleeps_levelsmap.csv')
        dailies_df = pd.read_csv('/content/drive/MyDrive/多變量專案/dailies.csv')
        epochs_df = pd.read_csv('/content/drive/MyDrive/多變量專案/epochs.csv', nrows=10000)  # 先讀取部分資料

        # 建立驗證器
        validator = GarminDataValidator()

        # 執行驗證
        print("\n開始資料品質驗證...\n")

        # 1. 驗證睡眠時間
        sleeps_validated = validator.validate_sleep_times(sleeps_df, sleeps_levels_df)

        # 2. 驗證睡眠階段
        stage_issues = validator.validate_sleep_stages(sleeps_df, sleeps_levels_df)

        # 3. 驗證日活動
        dailies_validated = validator.validate_daily_activity(dailies_df)

        # 4. 跨表驗證
        cross_validation = validator.cross_table_validation(sleeps_df, dailies_df, epochs_df)

        # 產生報告
        print("\n=== 驗證總結 ===")
        print("請檢查上述結果，並根據發現的問題制定資料清理策略")

    except FileNotFoundError as e:
        print(f"找不到檔案: {e}")
        print("請確保所有 CSV 檔案都在正確的路徑")
    except Exception as e:
        print(f"發生錯誤: {e}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

class GarminDataCleaner:
    """Garmin 資料清理與處理器 - 修正版"""

    def __init__(self):
        self.cleaning_log = []

    def analyze_date_issues(self, sleeps_df):
        """深入分析日期問題 - 修正版"""
        print("=== 日期問題深入分析 ===\n")

        # 確保時間欄位正確轉換
        sleeps_df['start_datetime'] = pd.to_datetime(sleeps_df['startTimeInSeconds'], unit='s')
        sleeps_df['end_datetime'] = sleeps_df['start_datetime'] + pd.to_timedelta(sleeps_df['durationInSeconds'], unit='s')
        sleeps_df['recorded_date'] = pd.to_datetime(sleeps_df['date'])

        # 計算各種日期
        sleeps_df['start_date'] = sleeps_df['start_datetime'].dt.date
        sleeps_df['end_date'] = sleeps_df['end_datetime'].dt.date
        sleeps_df['recorded_date_only'] = sleeps_df['recorded_date'].dt.date

        # 加入開始時間的小時
        sleeps_df['start_hour'] = sleeps_df['start_datetime'].dt.hour

        # 分析日期不一致的模式
        date_mismatch = sleeps_df[sleeps_df['recorded_date_only'] != sleeps_df['start_date']].copy()

        print(f"總記錄數: {len(sleeps_df)}")
        print(f"日期不一致記錄數: {len(date_mismatch)} ({len(date_mismatch)/len(sleeps_df)*100:.1f}%)\n")

        # 分析不一致的模式
        if len(date_mismatch) > 0:
            print("依睡眠開始時間分組的不一致記錄：")
            hour_analysis = date_mismatch.groupby('start_hour').size().sort_index()
            for hour, count in hour_analysis.items():
                print(f"  {hour:02d}:00 - {count} 筆")

            # 修正日期差異計算
            date_mismatch['date_diff'] = (
                pd.to_datetime(date_mismatch['recorded_date_only']) -
                pd.to_datetime(date_mismatch['start_date'])
            ).dt.days

            print(f"\n日期差異分析：")
            print(f"  記錄日期晚於開始日期 1 天: {sum(date_mismatch['date_diff'] == 1)} 筆")
            print(f"  記錄日期晚於開始日期 2+ 天: {sum(date_mismatch['date_diff'] >= 2)} 筆")
            print(f"  記錄日期早於開始日期: {sum(date_mismatch['date_diff'] < 0)} 筆")

            # 顯示一些範例
            print("\n不一致記錄範例：")
            sample_cols = ['_userId', 'recorded_date_only', 'start_datetime',
                          'end_datetime', 'start_hour', 'date_diff']
            print(date_mismatch[sample_cols].head(10))

            # 分析睡眠類型
            print("\n=== 睡眠類型分析 ===")
            # 根據開始時間和持續時間判斷睡眠類型
            date_mismatch['sleep_hours'] = date_mismatch['durationInSeconds'] / 3600

            # 午睡特徵：下午開始，持續時間較短
            afternoon_sleep = date_mismatch[(date_mismatch['start_hour'] >= 12) &
                                           (date_mismatch['start_hour'] < 20)]
            print(f"\n下午睡眠（12:00-20:00）統計：")
            print(f"  記錄數: {len(afternoon_sleep)}")
            print(f"  平均時長: {afternoon_sleep['sleep_hours'].mean():.2f} 小時")
            print(f"  中位數時長: {afternoon_sleep['sleep_hours'].median():.2f} 小時")

        return sleeps_df, date_mismatch

    def classify_sleep_types(self, sleeps_df):
        """分類睡眠類型"""
        print("\n=== 睡眠類型分類 ===\n")

        # 計算睡眠時長（小時）
        sleeps_df['sleep_hours'] = sleeps_df['durationInSeconds'] / 3600

        # 分類睡眠類型
        def classify_sleep(row):
            start_hour = row['start_hour']
            duration = row['sleep_hours']

            # 夜間主要睡眠：晚上開始或凌晨開始，且時長超過4小時
            if (start_hour >= 20 or start_hour < 8) and duration >= 4:
                return 'night_sleep'
            # 午睡：中午到傍晚開始，時長較短
            elif 11 <= start_hour < 16 and duration <= 3:
                return 'afternoon_nap'
            # 傍晚小睡：傍晚開始的短睡眠
            elif 16 <= start_hour < 20 and duration <= 3:
                return 'evening_nap'
            # 其他
            else:
                return 'other'

        sleeps_df['sleep_type'] = sleeps_df.apply(classify_sleep, axis=1)

        # 統計各類型
        print("睡眠類型分布：")
        type_stats = sleeps_df['sleep_type'].value_counts()
        for sleep_type, count in type_stats.items():
            print(f"  {sleep_type}: {count} ({count/len(sleeps_df)*100:.1f}%)")

        # 各類型的時長統計
        print("\n各睡眠類型的平均時長：")
        for sleep_type in sleeps_df['sleep_type'].unique():
            type_data = sleeps_df[sleeps_df['sleep_type'] == sleep_type]
            print(f"  {sleep_type}: {type_data['sleep_hours'].mean():.2f} 小時")

        return sleeps_df

    def create_smart_date_logic(self, sleeps_df):
        """建立更智慧的日期歸屬邏輯"""
        print("\n=== 智慧日期歸屬邏輯 ===\n")

        def smart_sleep_date(row):
            """根據睡眠類型決定日期歸屬"""
            sleep_type = row['sleep_type']

            if sleep_type == 'night_sleep':
                # 夜間睡眠：使用結束日期（醒來的日期）
                return row['end_date']
            elif sleep_type in ['afternoon_nap', 'evening_nap']:
                # 午睡和傍晚小睡：使用開始日期
                return row['start_date']
            else:
                # 其他情況：根據開始時間判斷
                if row['start_hour'] >= 18:
                    return row['end_date']
                else:
                    return row['start_date']

        sleeps_df['sleep_date_final'] = sleeps_df.apply(smart_sleep_date, axis=1)

        # 驗證新的日期邏輯
        matches_original = sum(sleeps_df['recorded_date_only'] == sleeps_df['sleep_date_final'])
        print(f"新邏輯與原始日期的匹配率: {matches_original/len(sleeps_df)*100:.1f}%")

        # 分析改變的記錄
        date_changed = sleeps_df[sleeps_df['recorded_date_only'] != sleeps_df['sleep_date_final']]
        if len(date_changed) > 0:
            print(f"\n需要調整日期的記錄: {len(date_changed)} 筆")
            print("主要是這些類型的睡眠：")
            print(date_changed['sleep_type'].value_counts())

        return sleeps_df

    def clean_and_integrate(self, sleeps_df, dailies_df):
        """清理並整合資料"""
        print("\n=== 資料清理與整合 ===\n")

        # 1. 標記異常睡眠時長
        MIN_SLEEP_HOURS = 0.5  # 最少30分鐘
        MAX_SLEEP_HOURS = 14.0

        sleeps_df['is_valid_duration'] = (
            (sleeps_df['sleep_hours'] >= MIN_SLEEP_HOURS) &
            (sleeps_df['sleep_hours'] <= MAX_SLEEP_HOURS)
        )

        # 2. 分別處理主要睡眠和午睡
        # 主要睡眠資料集
        main_sleep = sleeps_df[
            (sleeps_df['sleep_type'] == 'night_sleep') &
            (sleeps_df['is_valid_duration'])
        ].copy()

        # 午睡資料集（可選）
        naps = sleeps_df[
            (sleeps_df['sleep_type'].isin(['afternoon_nap', 'evening_nap'])) &
            (sleeps_df['is_valid_duration'])
        ].copy()

        print(f"有效的主要睡眠記錄: {len(main_sleep)}")
        print(f"有效的午睡記錄: {len(naps)}")

        # 3. 準備合併
        # 使用最終日期進行合併
        main_sleep['merge_date'] = main_sleep['sleep_date_final']
        dailies_df['merge_date'] = pd.to_datetime(dailies_df['date']).dt.date

        # 4. 合併主要睡眠與日活動
        merged_main = pd.merge(
            dailies_df,
            main_sleep[['_userId', 'merge_date', 'summaryId', 'sleep_hours',
                       'deepSleepDurationInSeconds', 'lightSleepDurationInSeconds',
                       'remSleepInSeconds', 'awakeDurationInSeconds']],
            on=['_userId', 'merge_date'],
            how='inner',
            suffixes=('', '_sleep')
        )

        print(f"\n成功配對的主要睡眠記錄: {len(merged_main)}")
        print(f"配對率: {len(merged_main)/len(main_sleep)*100:.1f}%")

        # 5. 計算額外的午睡資訊（如果需要）
        # 統計每天的午睡次數和總時長
        if len(naps) > 0:
            nap_summary = naps.groupby(['_userId', 'start_date']).agg({
                'sleep_hours': ['count', 'sum']
            }).reset_index()
            nap_summary.columns = ['_userId', 'date', 'nap_count', 'total_nap_hours']
            nap_summary['date'] = pd.to_datetime(nap_summary['date']).dt.date

            # 可選：將午睡資訊加入主資料集
            merged_with_naps = pd.merge(
                merged_main,
                nap_summary,
                left_on=['_userId', 'merge_date'],
                right_on=['_userId', 'date'],
                how='left'
            )
            merged_with_naps['nap_count'] = merged_with_naps['nap_count'].fillna(0)
            merged_with_naps['total_nap_hours'] = merged_with_naps['total_nap_hours'].fillna(0)

            print(f"\n有午睡記錄的天數: {sum(merged_with_naps['nap_count'] > 0)}")

        return merged_main, main_sleep, naps

# 執行修正後的清理流程
def run_fixed_cleaning():
    """執行修正後的清理流程"""
    print("開始修正後的資料清理流程...\n")

    try:
        # 讀取資料
        sleeps_df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleeps.csv')
        dailies_df = pd.read_csv('/content/drive/MyDrive/多變量專案/dailies.csv')

        # 建立清理器
        cleaner = GarminDataCleaner()

        # 1. 分析日期問題
        sleeps_df, date_issues = cleaner.analyze_date_issues(sleeps_df)

        # 2. 分類睡眠類型
        sleeps_df = cleaner.classify_sleep_types(sleeps_df)

        # 3. 建立智慧日期邏輯
        sleeps_df = cleaner.create_smart_date_logic(sleeps_df)

        # 4. 清理並整合資料
        merged_data, main_sleep, naps = cleaner.clean_and_integrate(sleeps_df, dailies_df)

        # 5. 儲存結果
        print("\n=== 儲存清理後的資料 ===")
        sleeps_df.to_csv('/content/drive/MyDrive/多變量專案/sleeps_with_classification.csv', index=False)
        main_sleep.to_csv('/content/drive/MyDrive/多變量專案/main_sleep_cleaned.csv', index=False)
        naps.to_csv('/content/drive/MyDrive/多變量專案/naps_cleaned.csv', index=False)
        merged_data.to_csv('/content/drive/MyDrive/多變量專案/sleep_activity_final.csv', index=False)

        print("\n資料清理完成！")
        print(f"最終整合資料集包含 {len(merged_data)} 筆記錄")
        print(f"涵蓋 {merged_data['_userId'].nunique()} 位使用者")

        # 產生簡單的品質報告
        print("\n=== 資料品質總結 ===")
        print(f"原始睡眠記錄: {len(sleeps_df)}")
        print(f"  - 主要睡眠: {sum(sleeps_df['sleep_type'] == 'night_sleep')}")
        print(f"  - 午睡: {sum(sleeps_df['sleep_type'] == 'afternoon_nap')}")
        print(f"  - 傍晚小睡: {sum(sleeps_df['sleep_type'] == 'evening_nap')}")
        print(f"  - 其他: {sum(sleeps_df['sleep_type'] == 'other')}")

        return sleeps_df, merged_data

    except Exception as e:
        print(f"處理過程中發生錯誤: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    sleeps_classified, final_dataset = run_fixed_cleaning()

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_sleep_patterns(sleeps_df):
    """深入分析睡眠模式以制定更好的分類策略"""
    print("=== 睡眠模式深入分析 ===\n")

    # 基礎轉換
    sleeps_df['start_datetime'] = pd.to_datetime(sleeps_df['startTimeInSeconds'], unit='s')
    sleeps_df['sleep_hours'] = sleeps_df['durationInSeconds'] / 3600
    sleeps_df['start_hour'] = sleeps_df['start_datetime'].dt.hour

    # 製作熱圖：開始時間 vs 睡眠時長
    print("開始時間與睡眠時長的分布：")

    # 建立交叉表
    hour_duration_table = pd.crosstab(
        sleeps_df['start_hour'],
        pd.cut(sleeps_df['sleep_hours'],
               bins=[0, 2, 4, 6, 8, 10, 24],
               labels=['0-2h', '2-4h', '4-6h', '6-8h', '8-10h', '10+h'])
    )

    print("\n各開始時間的睡眠時長分布：")
    print(hour_duration_table)

    # 分析主要模式
    print("\n=== 主要睡眠模式識別 ===")

    # 模式1：傳統夜間睡眠（20:00-02:00開始）
    traditional_night = sleeps_df[
        ((sleeps_df['start_hour'] >= 20) | (sleeps_df['start_hour'] <= 2)) &
        (sleeps_df['sleep_hours'] >= 4)
    ]

    # 模式2：早睡型（16:00-20:00開始，睡眠時長超過4小時）
    early_sleepers = sleeps_df[
        (sleeps_df['start_hour'] >= 16) &
        (sleeps_df['start_hour'] < 20) &
        (sleeps_df['sleep_hours'] >= 4)
    ]

    # 模式3：真正的午睡（10:00-16:00開始，睡眠時長小於3小時）
    real_naps = sleeps_df[
        (sleeps_df['start_hour'] >= 10) &
        (sleeps_df['start_hour'] < 16) &
        (sleeps_df['sleep_hours'] <= 3)
    ]

    print(f"傳統夜間睡眠（20:00-02:00開始，>4小時）: {len(traditional_night)} 筆")
    print(f"早睡型（16:00-20:00開始，>4小時）: {len(early_sleepers)} 筆")
    print(f"真正的午睡（10:00-16:00開始，<3小時）: {len(real_naps)} 筆")

    # 顯示早睡型的詳細統計
    if len(early_sleepers) > 0:
        print(f"\n早睡型詳細分析：")
        print(f"  平均睡眠時長: {early_sleepers['sleep_hours'].mean():.2f} 小時")
        print(f"  開始時間分布:")
        for hour in range(16, 20):
            count = sum(early_sleepers['start_hour'] == hour)
            if count > 0:
                print(f"    {hour}:00 - {count} 筆")

    return sleeps_df

def improved_sleep_classification(sleeps_df):
    """改進的睡眠分類函數"""
    print("\n=== 改進的睡眠分類 ===\n")

    def classify_sleep_v2(row):
        start_hour = row['start_hour']
        duration = row['sleep_hours']

        # 主要睡眠：時長超過4小時的都算（不限制開始時間）
        if duration >= 4:
            # 根據開始時間細分
            if start_hour >= 20 or start_hour <= 4:
                return 'night_sleep_traditional'  # 傳統夜間睡眠
            elif 16 <= start_hour < 20:
                return 'night_sleep_early'  # 早睡型
            elif 4 < start_hour < 10:
                return 'night_sleep_late'  # 晚起型（可能是夜班工作者）
            else:
                return 'long_day_sleep'  # 白天長睡眠（特殊情況）

        # 短睡眠：時長小於4小時
        else:
            if 10 <= start_hour < 16:
                return 'afternoon_nap'  # 午睡
            elif 16 <= start_hour < 20:
                return 'evening_nap'  # 傍晚小睡
            elif start_hour >= 20 or start_hour <= 4:
                return 'short_night_sleep'  # 夜間短睡眠
            else:
                return 'other_short_sleep'  # 其他短睡眠

    sleeps_df['sleep_type_v2'] = sleeps_df.apply(classify_sleep_v2, axis=1)

    # 統計新分類
    print("改進後的睡眠類型分布：")
    type_stats = sleeps_df['sleep_type_v2'].value_counts()
    for sleep_type, count in type_stats.items():
        avg_duration = sleeps_df[sleeps_df['sleep_type_v2'] == sleep_type]['sleep_hours'].mean()
        print(f"{sleep_type:25s}: {count:4d} ({count/len(sleeps_df)*100:5.1f}%) - 平均 {avg_duration:.2f} 小時")

    # 計算主要睡眠的總比例
    main_sleep_types = ['night_sleep_traditional', 'night_sleep_early',
                       'night_sleep_late', 'long_day_sleep']
    main_sleep_count = sum(sleeps_df['sleep_type_v2'].isin(main_sleep_types))
    print(f"\n主要睡眠總計: {main_sleep_count} ({main_sleep_count/len(sleeps_df)*100:.1f}%)")

    return sleeps_df

def create_final_date_logic(sleeps_df):
    """基於改進的分類建立最終的日期邏輯"""
    print("\n=== 最終日期歸屬邏輯 ===\n")

    def final_sleep_date(row):
        sleep_type = row['sleep_type_v2']

        # 所有主要睡眠類型：使用結束日期（醒來的日期）
        if sleep_type in ['night_sleep_traditional', 'night_sleep_early',
                         'night_sleep_late', 'long_day_sleep']:
            return row['end_date']

        # 所有短睡眠（午睡、小睡等）：使用開始日期
        else:
            return row['start_date']

    # 需要先計算 end_date
    sleeps_df['end_datetime'] = sleeps_df['start_datetime'] + pd.to_timedelta(sleeps_df['durationInSeconds'], unit='s')
    sleeps_df['start_date'] = sleeps_df['start_datetime'].dt.date
    sleeps_df['end_date'] = sleeps_df['end_datetime'].dt.date
    sleeps_df['recorded_date'] = pd.to_datetime(sleeps_df['date']).dt.date

    sleeps_df['sleep_date_final'] = sleeps_df.apply(final_sleep_date, axis=1)

    # 驗證匹配率
    matches = sum(sleeps_df['recorded_date'] == sleeps_df['sleep_date_final'])
    print(f"最終日期邏輯與原始記錄的匹配率: {matches/len(sleeps_df)*100:.1f}%")

    # 分析需要調整的記錄
    mismatches = sleeps_df[sleeps_df['recorded_date'] != sleeps_df['sleep_date_final']]
    if len(mismatches) > 0:
        print(f"\n需要調整日期的記錄: {len(mismatches)} 筆")
        print("主要是這些類型：")
        print(mismatches['sleep_type_v2'].value_counts())

    return sleeps_df

def create_analysis_dataset(sleeps_df, dailies_df):
    """建立最終的分析資料集"""
    print("\n=== 建立最終分析資料集 ===\n")

    # 1. 篩選主要睡眠
    main_sleep_types = ['night_sleep_traditional', 'night_sleep_early',
                       'night_sleep_late', 'long_day_sleep']

    main_sleep = sleeps_df[
        sleeps_df['sleep_type_v2'].isin(main_sleep_types)
    ].copy()

    print(f"主要睡眠記錄數: {len(main_sleep)}")

    # 2. 準備合併
    main_sleep['merge_date'] = main_sleep['sleep_date_final']
    dailies_df['merge_date'] = pd.to_datetime(dailies_df['date']).dt.date

    # 3. 執行合併
    merged = pd.merge(
        dailies_df,
        main_sleep[['_userId', 'merge_date', 'summaryId', 'sleep_hours',
                   'sleep_type_v2', 'deepSleepDurationInSeconds',
                   'lightSleepDurationInSeconds', 'remSleepInSeconds']],
        on=['_userId', 'merge_date'],
        how='inner'
    )

    print(f"成功配對的記錄: {len(merged)}")
    print(f"配對率: {len(merged)/len(main_sleep)*100:.1f}%")
    print(f"涵蓋使用者數: {merged['_userId'].nunique()}")

    # 4. 加入午睡資訊（如果需要）
    nap_types = ['afternoon_nap', 'evening_nap']
    naps = sleeps_df[sleeps_df['sleep_type_v2'].isin(nap_types)]

    if len(naps) > 0:
        nap_summary = naps.groupby(['_userId', 'start_date']).agg({
            'sleep_hours': ['count', 'sum']
        }).reset_index()
        nap_summary.columns = ['_userId', 'date', 'nap_count', 'total_nap_hours']

        merged_with_naps = pd.merge(
            merged,
            nap_summary,
            left_on=['_userId', 'merge_date'],
            right_on=['_userId', 'date'],
            how='left'
        )
        merged_with_naps['nap_count'] = merged_with_naps['nap_count'].fillna(0)
        merged_with_naps['total_nap_hours'] = merged_with_naps['total_nap_hours'].fillna(0)

        print(f"\n有午睡記錄的天數: {sum(merged_with_naps['nap_count'] > 0)}")

        return merged_with_naps

    return merged

# 執行改進的分析
def run_improved_analysis():
    """執行改進的分析流程"""
    print("開始改進的睡眠分析流程...\n")

    try:
        # 讀取資料
        sleeps_df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleeps.csv')
        dailies_df = pd.read_csv('/content/drive/MyDrive/多變量專案/dailies.csv')

        # 1. 深入分析睡眠模式
        sleeps_df = analyze_sleep_patterns(sleeps_df)

        # 2. 使用改進的分類
        sleeps_df = improved_sleep_classification(sleeps_df)

        # 3. 建立最終日期邏輯
        sleeps_df = create_final_date_logic(sleeps_df)

        # 4. 建立分析資料集
        final_dataset = create_analysis_dataset(sleeps_df, dailies_df)

        # 5. 儲存結果
        print("\n=== 儲存結果 ===")
        sleeps_df.to_csv('/content/drive/MyDrive/多變量專案/sleeps_improved_classification.csv', index=False)
        final_dataset.to_csv('/content/drive/MyDrive/多變量專案/sleep_activity_final_v2.csv', index=False)

        print("\n分析完成！")

        # 6. 產生摘要報告
        print("\n=== 最終資料集摘要 ===")
        print(f"總睡眠記錄: {len(sleeps_df)}")
        print(f"主要睡眠記錄: {sum(sleeps_df['sleep_type_v2'].str.contains('night_sleep|long_day'))}")
        print(f"最終分析資料集: {len(final_dataset)} 筆")
        print(f"資料時間範圍: {final_dataset['merge_date'].min()} 至 {final_dataset['merge_date'].max()}")

        return sleeps_df, final_dataset

    except Exception as e:
        print(f"處理過程中發生錯誤: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    sleeps_improved, final_data = run_improved_analysis()

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def diagnose_pairing_issue(sleeps_df, dailies_df):
    """診斷配對問題"""
    print("=== 配對問題診斷 ===\n")

    # 1. 檢查是否有重複的睡眠記錄
    sleep_date_count = sleeps_df.groupby(['_userId', 'sleep_date_final']).size()
    duplicates = sleep_date_count[sleep_date_count > 1]

    if len(duplicates) > 0:
        print(f"發現 {len(duplicates)} 個使用者-日期組合有多筆睡眠記錄")
        print("\n範例（同一天多筆睡眠）：")
        print(duplicates.head(10))

        # 顯示具體案例
        sample_user, sample_date = duplicates.index[0]
        sample_records = sleeps_df[
            (sleeps_df['_userId'] == sample_user) &
            (sleeps_df['sleep_date_final'] == sample_date)
        ]
        print(f"\n使用者 {sample_user} 在 {sample_date} 的睡眠記錄：")
        print(sample_records[['start_datetime', 'sleep_hours', 'sleep_type_v2']])

    # 2. 檢查日活動資料是否有重複
    daily_date_count = dailies_df.groupby(['_userId', 'date']).size()
    daily_duplicates = daily_date_count[daily_date_count > 1]

    if len(daily_duplicates) > 0:
        print(f"\n發現 {len(daily_duplicates)} 個使用者-日期組合有多筆日活動記錄")

    return duplicates

def create_deduped_sleep_dataset(sleeps_df):
    """建立去重的睡眠資料集"""
    print("\n=== 建立去重的睡眠資料集 ===\n")

    # 只保留主要睡眠
    main_sleep_types = ['night_sleep_traditional', 'night_sleep_early',
                       'night_sleep_late', 'long_day_sleep']
    main_sleep = sleeps_df[sleeps_df['sleep_type_v2'].isin(main_sleep_types)].copy()

    # 對於同一天有多筆主要睡眠的情況，選擇最長的一筆
    def select_primary_sleep(group):
        """選擇一天中最主要的睡眠記錄"""
        # 優先選擇時長最長的
        return group.nlargest(1, 'sleep_hours')

    main_sleep_deduped = main_sleep.groupby(['_userId', 'sleep_date_final']).apply(
        select_primary_sleep
    ).reset_index(drop=True)

    print(f"原始主要睡眠記錄: {len(main_sleep)}")
    print(f"去重後主要睡眠記錄: {len(main_sleep_deduped)}")
    print(f"移除的重複記錄: {len(main_sleep) - len(main_sleep_deduped)}")

    return main_sleep_deduped

def analyze_sleep_patterns_by_user(sleeps_df):
    """按使用者分析睡眠模式"""
    print("\n=== 使用者睡眠模式分析 ===\n")

    # 計算每個使用者的主要睡眠類型
    user_sleep_patterns = sleeps_df.groupby('_userId')['sleep_type_v2'].agg([
        lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown',  # 最常見的睡眠類型
        'count'
    ])
    user_sleep_patterns.columns = ['dominant_pattern', 'total_records']

    # 統計各種睡眠模式的使用者數
    pattern_users = user_sleep_patterns['dominant_pattern'].value_counts()
    print("使用者的主要睡眠模式分布：")
    for pattern, count in pattern_users.items():
        print(f"  {pattern}: {count} 位使用者")

    # 找出特殊案例
    print("\n=== 特殊使用者案例 ===")

    # 白天睡眠為主的使用者
    day_sleepers = user_sleep_patterns[
        user_sleep_patterns['dominant_pattern'] == 'long_day_sleep'
    ]
    if len(day_sleepers) > 0:
        print(f"\n主要在白天睡覺的使用者: {len(day_sleepers)} 位")
        print("範例：")
        print(day_sleepers.head())

    return user_sleep_patterns

def create_final_merged_dataset(main_sleep_deduped, dailies_df):
    """建立最終的合併資料集"""
    print("\n=== 建立最終合併資料集 ===\n")

    # 準備合併鍵
    main_sleep_deduped['merge_date'] = main_sleep_deduped['sleep_date_final']
    dailies_df['merge_date'] = pd.to_datetime(dailies_df['date']).dt.date

    # 去重日活動資料（如果有重複）
    dailies_deduped = dailies_df.drop_duplicates(subset=['_userId', 'merge_date'])

    # 執行合併
    merged = pd.merge(
        dailies_deduped,
        main_sleep_deduped[['_userId', 'merge_date', 'summaryId', 'sleep_hours',
                           'sleep_type_v2', 'deepSleepDurationInSeconds',
                           'lightSleepDurationInSeconds', 'remSleepInSeconds']],
        on=['_userId', 'merge_date'],
        how='inner'
    )

    print(f"日活動記錄（去重後）: {len(dailies_deduped)}")
    print(f"睡眠記錄（去重後）: {len(main_sleep_deduped)}")
    print(f"成功配對記錄: {len(merged)}")
    print(f"配對率: {len(merged)/len(main_sleep_deduped)*100:.1f}%")

    # 分析未配對的原因
    unmatched_sleep = main_sleep_deduped[
        ~main_sleep_deduped.set_index(['_userId', 'merge_date']).index.isin(
            merged.set_index(['_userId', 'merge_date']).index
        )
    ]

    if len(unmatched_sleep) > 0:
        print(f"\n未配對的睡眠記錄: {len(unmatched_sleep)}")
        print("可能原因：對應日期沒有活動資料")

    return merged

def generate_analysis_report(sleeps_df, merged_df, user_patterns):
    """生成分析報告"""
    print("\n=== 最終分析報告 ===\n")

    print("1. 資料概況")
    print(f"   - 總使用者數: {sleeps_df['_userId'].nunique()}")
    print(f"   - 資料時間範圍: {merged_df['merge_date'].min()} 至 {merged_df['merge_date'].max()}")
    print(f"   - 最終分析記錄: {len(merged_df)}")

    print("\n2. 睡眠模式發現")
    print("   - 58.4% 的記錄是早睡型（16:00-20:00開始）")
    print("   - 35.7% 的記錄是白天長睡眠")
    print("   - 傳統夜間睡眠只佔 1.8%")

    print("\n3. 可能的解釋")
    print("   a) 時區問題：資料可能未正確調整時區")
    print("   b) 特殊族群：可能包含輪班工作者或特殊作息者")
    print("   c) 地區差異：某些地區可能有午後長睡習慣")

    print("\n4. 建議")
    print("   - 在分析時考慮按睡眠類型分群")
    print("   - 可能需要調查使用者背景資訊")
    print("   - 考慮時區調整的必要性")

# 執行完整的診斷和修正流程
def run_complete_analysis():
    """執行完整的分析流程"""
    print("執行完整的配對診斷和修正...\n")

    try:
        # 讀取之前保存的分類結果
        sleeps_df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleeps_improved_classification.csv')
        dailies_df = pd.read_csv('/content/drive/MyDrive/多變量專案/dailies.csv')

        # 轉換日期欄位
        date_columns = ['start_date', 'end_date', 'recorded_date', 'sleep_date_final']
        for col in date_columns:
            if col in sleeps_df.columns:
                sleeps_df[col] = pd.to_datetime(sleeps_df[col]).dt.date

        # 1. 診斷配對問題
        duplicates = diagnose_pairing_issue(sleeps_df, dailies_df)

        # 2. 分析使用者睡眠模式
        user_patterns = analyze_sleep_patterns_by_user(sleeps_df)

        # 3. 建立去重的睡眠資料集
        main_sleep_deduped = create_deduped_sleep_dataset(sleeps_df)

        # 4. 建立最終合併資料集
        final_merged = create_final_merged_dataset(main_sleep_deduped, dailies_df)

        # 5. 生成報告
        generate_analysis_report(sleeps_df, final_merged, user_patterns)

        # 6. 儲存最終結果
        print("\n儲存最終資料集...")
        main_sleep_deduped.to_csv('main_sleep_deduped.csv', index=False)
        final_merged.to_csv('sleep_activity_final_clean.csv', index=False)
        user_patterns.to_csv('user_sleep_patterns.csv')

        return final_merged, user_patterns

    except Exception as e:
        print(f"處理過程中發生錯誤: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    final_data, user_analysis = run_complete_analysis()

# **台灣時間GMT+8調整**:

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_timezone_offset(sleeps_df):
    """分析時區偏移量"""
    print("=== 時區偏移分析 ===\n")

    # 檢查 startTimeOffsetInSeconds 欄位
    if 'startTimeOffsetInSeconds' in sleeps_df.columns:
        offset_values = sleeps_df['startTimeOffsetInSeconds'].value_counts()
        print("時區偏移值分布：")
        for offset, count in offset_values.items():
            hours = offset / 3600
            print(f"  {offset} 秒 (GMT{hours:+.0f}): {count} 筆記錄")

        # 計算最常見的時區
        most_common_offset = sleeps_df['startTimeOffsetInSeconds'].mode()[0]
        most_common_hours = most_common_offset / 3600
        print(f"\n最常見的時區偏移：GMT{most_common_hours:+.0f}")
    else:
        print("注意：資料中沒有 startTimeOffsetInSeconds 欄位")
        print("假設所有資料都需要 +8 小時調整（台灣時區）")
        return None

    return sleeps_df

def apply_timezone_correction(sleeps_df, default_offset_hours=8):
    """應用時區修正"""
    print(f"\n=== 應用時區修正 (預設 GMT+{default_offset_hours}) ===\n")

    # 原始時間（可能是 UTC）
    sleeps_df['start_datetime_utc'] = pd.to_datetime(sleeps_df['startTimeInSeconds'], unit='s')

    # 應用時區修正
    if 'startTimeOffsetInSeconds' in sleeps_df.columns:
        # 使用資料中的偏移值
        sleeps_df['timezone_offset_hours'] = sleeps_df['startTimeOffsetInSeconds'] / 3600
        sleeps_df['start_datetime_local'] = sleeps_df.apply(
            lambda row: row['start_datetime_utc'] + timedelta(seconds=row['startTimeOffsetInSeconds']),
            axis=1
        )
    else:
        # 使用預設偏移值（假設台灣時區）
        sleeps_df['timezone_offset_hours'] = default_offset_hours
        sleeps_df['start_datetime_local'] = sleeps_df['start_datetime_utc'] + timedelta(hours=default_offset_hours)

    # 重新計算本地時間的小時
    sleeps_df['start_hour_utc'] = sleeps_df['start_datetime_utc'].dt.hour
    sleeps_df['start_hour_local'] = sleeps_df['start_datetime_local'].dt.hour

    # 比較調整前後的分布
    print("時間調整前後比較：")
    print("\nUTC 時間分布（調整前）：")
    utc_dist = sleeps_df['start_hour_utc'].value_counts().sort_index()
    for hour, count in utc_dist.items():
        if count > 100:  # 只顯示主要時段
            print(f"  {hour:02d}:00 - {count} 筆")

    print("\n本地時間分布（調整後）：")
    local_dist = sleeps_df['start_hour_local'].value_counts().sort_index()
    for hour, count in local_dist.items():
        if count > 100:  # 只顯示主要時段
            print(f"  {hour:02d}:00 - {count} 筆")

    return sleeps_df

def reclassify_sleep_with_timezone(sleeps_df):
    """使用調整後的時間重新分類睡眠"""
    print("\n=== 時區調整後的睡眠分類 ===\n")

    sleeps_df['sleep_hours'] = sleeps_df['durationInSeconds'] / 3600

    def classify_sleep_corrected(row):
        start_hour = row['start_hour_local']  # 使用本地時間
        duration = row['sleep_hours']

        # 主要睡眠：時長超過4小時
        if duration >= 4:
            if 20 <= start_hour or start_hour <= 4:
                return 'night_sleep_normal'  # 正常夜間睡眠
            elif 4 < start_hour < 10:
                return 'morning_sleep'  # 早晨睡眠（可能是夜班）
            elif 10 <= start_hour < 16:
                return 'day_sleep'  # 白天睡眠
            else:  # 16-20
                return 'evening_sleep'  # 傍晚睡眠

        # 短睡眠：時長小於4小時
        else:
            if 11 <= start_hour < 16:
                return 'afternoon_nap'  # 午睡
            elif 16 <= start_hour < 20:
                return 'evening_nap'  # 傍晚小睡
            else:
                return 'short_sleep'  # 其他短睡眠

    sleeps_df['sleep_type_corrected'] = sleeps_df.apply(classify_sleep_corrected, axis=1)

    # 統計新分類
    print("調整後的睡眠類型分布：")
    type_stats = sleeps_df['sleep_type_corrected'].value_counts()
    for sleep_type, count in type_stats.items():
        percentage = count / len(sleeps_df) * 100
        avg_duration = sleeps_df[sleeps_df['sleep_type_corrected'] == sleep_type]['sleep_hours'].mean()
        print(f"{sleep_type:20s}: {count:4d} ({percentage:5.1f}%) - 平均 {avg_duration:.2f} 小時")

    # 計算正常夜間睡眠的比例
    normal_sleep_count = sum(sleeps_df['sleep_type_corrected'] == 'night_sleep_normal')
    normal_sleep_pct = normal_sleep_count / len(sleeps_df) * 100
    print(f"\n正常夜間睡眠比例：{normal_sleep_pct:.1f}% （調整前：1.8%）")

    return sleeps_df

def create_comparison_visualization(sleeps_df):
    """創建調整前後的比較視覺化"""
    print("\n=== 生成比較圖表 ===\n")

    # 創建比較資料
    comparison_data = []

    # 調整前的分布（假設原始分類）
    if 'sleep_type_v2' in sleeps_df.columns:
        before_dist = sleeps_df['sleep_type_v2'].value_counts()
        for sleep_type, count in before_dist.items():
            comparison_data.append({
                'adjustment': '調整前',
                'sleep_type': sleep_type,
                'count': count,
                'percentage': count / len(sleeps_df) * 100
            })

    # 調整後的分布
    after_dist = sleeps_df['sleep_type_corrected'].value_counts()
    for sleep_type, count in after_dist.items():
        comparison_data.append({
            'adjustment': '調整後',
            'sleep_type': sleep_type,
            'count': count,
            'percentage': count / len(sleeps_df) * 100
        })

    comparison_df = pd.DataFrame(comparison_data)

    # 輸出比較表格
    print("調整前後比較：")
    print(comparison_df.pivot_table(
        index='sleep_type',
        columns='adjustment',
        values='percentage',
        fill_value=0
    ).round(1))

    return comparison_df

def analyze_user_timezones(sleeps_df):
    """分析不同使用者的時區情況"""
    print("\n=== 使用者時區分析 ===\n")

    if 'startTimeOffsetInSeconds' in sleeps_df.columns:
        user_timezones = sleeps_df.groupby('_userId')['startTimeOffsetInSeconds'].agg(['mean', 'std', 'nunique'])

        # 找出時區不一致的使用者
        inconsistent_users = user_timezones[user_timezones['nunique'] > 1]
        if len(inconsistent_users) > 0:
            print(f"發現 {len(inconsistent_users)} 位使用者有多個時區記錄（可能旅行）")
            print("\n範例：")
            print(inconsistent_users.head())

        # 統計主要時區
        user_main_timezone = sleeps_df.groupby('_userId')['timezone_offset_hours'].agg(lambda x: x.mode()[0])
        timezone_dist = user_main_timezone.value_counts()

        print("\n使用者主要時區分布：")
        for tz, count in timezone_dist.items():
            print(f"  GMT{tz:+.0f}: {count} 位使用者")

    return sleeps_df

def create_corrected_dataset(sleeps_df, dailies_df):
    """創建時區修正後的最終資料集"""
    print("\n=== 創建修正後的資料集 ===\n")

    # 重新計算日期（使用本地時間）
    sleeps_df['end_datetime_local'] = sleeps_df['start_datetime_local'] + pd.to_timedelta(sleeps_df['durationInSeconds'], unit='s')
    sleeps_df['start_date_local'] = sleeps_df['start_datetime_local'].dt.date
    sleeps_df['end_date_local'] = sleeps_df['end_datetime_local'].dt.date

    # 使用修正後的邏輯決定睡眠日期
    def corrected_sleep_date(row):
        sleep_type = row['sleep_type_corrected']

        # 正常夜間睡眠：使用結束日期（醒來的日期）
        if sleep_type in ['night_sleep_normal', 'morning_sleep']:
            return row['end_date_local']
        # 其他：使用開始日期
        else:
            return row['start_date_local']

    sleeps_df['sleep_date_corrected'] = sleeps_df.apply(corrected_sleep_date, axis=1)

    # 比較修正前後的日期匹配
    if 'date' in sleeps_df.columns:
        sleeps_df['recorded_date'] = pd.to_datetime(sleeps_df['date']).dt.date
        matches = sum(sleeps_df['recorded_date'] == sleeps_df['sleep_date_corrected'])
        match_rate = matches / len(sleeps_df) * 100
        print(f"時區修正後的日期匹配率：{match_rate:.1f}%")

    # 只保留主要睡眠
    main_sleep_types = ['night_sleep_normal', 'morning_sleep', 'day_sleep', 'evening_sleep']
    main_sleep = sleeps_df[sleeps_df['sleep_type_corrected'].isin(main_sleep_types)].copy()

    print(f"\n主要睡眠記錄數：{len(main_sleep)}")
    print(f"其中正常夜間睡眠：{sum(main_sleep['sleep_type_corrected'] == 'night_sleep_normal')}")

    return sleeps_df, main_sleep

# 主執行函數
def run_timezone_analysis():
    """執行時區分析和修正"""
    print("開始時區分析和修正流程...\n")

    try:
        # 讀取資料
        sleeps_df = pd.read_csv('/content/drive/MyDrive/多變量專案/sleeps.csv')
        dailies_df = pd.read_csv('/content/drive/MyDrive/多變量專案/dailies.csv')

        print(f"載入 {len(sleeps_df)} 筆睡眠記錄\n")

        # 1. 分析時區偏移
        sleeps_df = analyze_timezone_offset(sleeps_df)

        # 2. 應用時區修正
        sleeps_df = apply_timezone_correction(sleeps_df, default_offset_hours=8)

        # 3. 重新分類睡眠
        sleeps_df = reclassify_sleep_with_timezone(sleeps_df)

        # 4. 分析使用者時區
        sleeps_df = analyze_user_timezones(sleeps_df)

        # 5. 創建比較視覺化
        comparison_df = create_comparison_visualization(sleeps_df)

        # 6. 創建修正後的資料集
        sleeps_corrected, main_sleep_corrected = create_corrected_dataset(sleeps_df, dailies_df)

        # 7. 儲存結果
        print("\n=== 儲存結果 ===")
        sleeps_corrected.to_csv('/content/drive/MyDrive/多變量專案/sleeps_timezone_corrected.csv', index=False)
        comparison_df.to_csv('/content/drive/MyDrive/多變量專案/timezone_correction_comparison.csv', index=False)

        print("\n時區分析完成！")
        print("\n=== 關鍵發現總結 ===")
        print("1. 如果資料確實是 UTC 時間且使用者在 GMT+8 時區")
        print("2. 大部分「下午」睡眠實際上是正常的夜間睡眠")
        print("3. 這將大幅改善資料的合理性")

        return sleeps_corrected, comparison_df

    except Exception as e:
        print(f"處理過程中發生錯誤：{e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    sleeps_tz_corrected, comparison = run_timezone_analysis()

### 時區偏移的識別與修正

初步探索性分析揭示了一個令人困惑的現象：63.3% 的睡眠記錄
顯示使用者在下午時段（UTC 16:00-20:00）開始睡眠，這與正常
的生理作息規律嚴重不符。

進一步調查 startTimeOffsetInSeconds 欄位發現，所有 3,976 筆
記錄均顯示 28,800 秒的偏移量，相當於 GMT+8 時區。這一發現
提供了關鍵線索：原始資料可能是以 UTC 時間記錄，而使用者
實際位於東亞地區（如台灣、中國、香港或新加坡）。

應用時區修正後，睡眠模式分布發生了戲劇性的轉變：
- 正常夜間睡眠（22:00-04:00）：從 1.8% 提升至 95.5%
- 異常日間睡眠：從 94.1% 降至 0.5%

此一發現不僅解決了資料的合理性問題，更突顯了在跨時區資料
分析中考慮時間標準化的重要性。

# 修正時區後，進行完整的資料處理

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

class GarminDataCompleteCleaner:
    """Garmin 資料完整清理器 - 包含所有清理步驟"""

    def __init__(self):
        self.cleaning_log = []
        self.stats = {}

    def run_complete_cleaning(self, sleeps_path, dailies_path):
        """執行完整的資料清理流程"""
        print("=== 開始完整的資料清理流程 ===\n")

        # 1. 讀取原始資料
        print("步驟 1: 讀取原始資料")
        sleeps_df = pd.read_csv(sleeps_path)
        dailies_df = pd.read_csv(dailies_path)
        self.stats['原始睡眠記錄'] = len(sleeps_df)
        self.stats['原始活動記錄'] = len(dailies_df)
        print(f"  - 睡眠記錄: {len(sleeps_df)} 筆")
        print(f"  - 活動記錄: {len(dailies_df)} 筆")

        # 2. 時區修正
        print("\n步驟 2: 時區修正")
        sleeps_df = self.apply_timezone_correction(sleeps_df)

        # 3. 異常值處理
        print("\n步驟 3: 異常值處理")
        sleeps_df = self.handle_outliers(sleeps_df)

        # 4. 去重處理
        print("\n步驟 4: 去重處理")
        sleeps_df = self.remove_duplicates(sleeps_df)

        # 5. 日期歸屬邏輯統一
        print("\n步驟 5: 日期歸屬邏輯統一")
        sleeps_df = self.unify_date_attribution(sleeps_df)

        # 6. 資料配對
        print("\n步驟 6: 睡眠-活動資料配對")
        merged_df = self.merge_sleep_activity(sleeps_df, dailies_df)

        # 7. 生成清理報告
        self.generate_cleaning_report()

        return sleeps_df, merged_df

    def apply_timezone_correction(self, sleeps_df):
        """時區修正（已有程式碼的簡化版）"""
        # 轉換時間戳記
        sleeps_df['start_datetime_utc'] = pd.to_datetime(sleeps_df['startTimeInSeconds'], unit='s')

        # 應用 GMT+8 修正
        if 'startTimeOffsetInSeconds' in sleeps_df.columns:
            sleeps_df['start_datetime_local'] = sleeps_df.apply(
                lambda row: row['start_datetime_utc'] + timedelta(seconds=row['startTimeOffsetInSeconds']),
                axis=1
            )
        else:
            sleeps_df['start_datetime_local'] = sleeps_df['start_datetime_utc'] + timedelta(hours=8)

        sleeps_df['start_hour_local'] = sleeps_df['start_datetime_local'].dt.hour

        # 重新分類睡眠類型
        sleeps_df['sleep_hours'] = sleeps_df['durationInSeconds'] / 3600
        sleeps_df['sleep_type'] = sleeps_df.apply(self.classify_sleep, axis=1)

        # 統計
        normal_sleep = sum(sleeps_df['sleep_type'] == 'night_sleep_normal')
        self.stats['時區修正後正常睡眠'] = normal_sleep
        print(f"  - 正常夜間睡眠: {normal_sleep} 筆 ({normal_sleep/len(sleeps_df)*100:.1f}%)")

        return sleeps_df

    def classify_sleep(self, row):
        """分類睡眠類型"""
        start_hour = row['start_hour_local']
        duration = row['sleep_hours']

        if duration >= 4:
            if 20 <= start_hour or start_hour <= 4:
                return 'night_sleep_normal'
            elif 4 < start_hour < 10:
                return 'morning_sleep'
            elif 10 <= start_hour < 16:
                return 'day_sleep'
            else:
                return 'evening_sleep'
        else:
            if 11 <= start_hour < 16:
                return 'afternoon_nap'
            else:
                return 'short_sleep'

    def handle_outliers(self, sleeps_df):
        """處理異常值"""
        initial_count = len(sleeps_df)

        # 定義異常值標準
        MIN_SLEEP_HOURS = 0.5  # 30分鐘
        MAX_SLEEP_HOURS = 14   # 14小時

        # 標記異常值
        sleeps_df['is_outlier'] = ~sleeps_df['sleep_hours'].between(MIN_SLEEP_HOURS, MAX_SLEEP_HOURS)

        # 統計異常值
        outlier_count = sleeps_df['is_outlier'].sum()
        print(f"  - 發現異常值: {outlier_count} 筆")
        print(f"    - 過短 (<30分鐘): {sum(sleeps_df['sleep_hours'] < MIN_SLEEP_HOURS)} 筆")
        print(f"    - 過長 (>14小時): {sum(sleeps_df['sleep_hours'] > MAX_SLEEP_HOURS)} 筆")

        # 特別調查20小時睡眠
        extreme_sleep = sleeps_df[sleeps_df['sleep_hours'] >= 20]
        if len(extreme_sleep) > 0:
            print(f"    - 極端案例 (≥20小時): {len(extreme_sleep)} 筆")
            print(f"      涉及使用者: {extreme_sleep['_userId'].unique()}")

        # 移除異常值
        sleeps_df = sleeps_df[~sleeps_df['is_outlier']].copy()
        self.stats['異常值移除數'] = outlier_count
        print(f"  - 移除後剩餘: {len(sleeps_df)} 筆")

        return sleeps_df

    def remove_duplicates(self, sleeps_df):
        """去重處理 - 保留最長睡眠"""
        initial_count = len(sleeps_df)

        # 計算結束時間和日期
        sleeps_df['end_datetime_local'] = sleeps_df['start_datetime_local'] + pd.to_timedelta(sleeps_df['durationInSeconds'], unit='s')
        sleeps_df['sleep_date_for_dedup'] = sleeps_df['end_datetime_local'].dt.date

        # 檢查重複
        duplicate_check = sleeps_df.groupby(['_userId', 'sleep_date_for_dedup']).size()
        duplicates = duplicate_check[duplicate_check > 1]

        print(f"  - 發現 {len(duplicates)} 個使用者-日期組合有重複記錄")

        # 對每個使用者-日期組合，保留最長的睡眠
        def keep_longest_sleep(group):
            # 如果只有一筆，直接返回
            if len(group) == 1:
                return group

            # 優先保留主要睡眠類型
            main_types = ['night_sleep_normal', 'morning_sleep', 'evening_sleep', 'day_sleep']
            main_sleep = group[group['sleep_type'].isin(main_types)]

            if len(main_sleep) > 0:
                # 在主要睡眠中選最長的
                return main_sleep.nlargest(1, 'sleep_hours')
            else:
                # 否則就選最長的
                return group.nlargest(1, 'sleep_hours')

        # 執行去重
        sleeps_df = sleeps_df.groupby(['_userId', 'sleep_date_for_dedup']).apply(
            keep_longest_sleep
        ).reset_index(drop=True)

        removed_count = initial_count - len(sleeps_df)
        self.stats['去重移除數'] = removed_count
        print(f"  - 移除重複記錄: {removed_count} 筆")
        print(f"  - 去重後剩餘: {len(sleeps_df)} 筆")

        return sleeps_df

    def unify_date_attribution(self, sleeps_df):
        """統一日期歸屬邏輯"""
        print("  - 應用統一的日期歸屬邏輯")

        def determine_sleep_date(row):
            """決定睡眠應歸屬的日期"""
            sleep_type = row['sleep_type']

            # 主要睡眠（≥4小時）：使用結束日期（醒來的日期）
            if sleep_type in ['night_sleep_normal', 'morning_sleep', 'evening_sleep', 'day_sleep']:
                return row['end_datetime_local'].date()
            # 短睡眠（<4小時）：使用開始日期
            else:
                return row['start_datetime_local'].date()

        sleeps_df['sleep_date_final'] = sleeps_df.apply(determine_sleep_date, axis=1)

        # 檢查與原始日期的匹配率
        if 'date' in sleeps_df.columns:
            sleeps_df['original_date'] = pd.to_datetime(sleeps_df['date']).dt.date
            matches = sum(sleeps_df['original_date'] == sleeps_df['sleep_date_final'])
            match_rate = matches / len(sleeps_df) * 100
            self.stats['日期匹配率'] = match_rate
            print(f"  - 統一邏輯後的日期匹配率: {match_rate:.1f}%")

        return sleeps_df

    def merge_sleep_activity(self, sleeps_df, dailies_df):
        """睡眠-活動資料配對整合"""
        # 只保留主要睡眠用於配對
        main_sleep_types = ['night_sleep_normal', 'morning_sleep', 'evening_sleep', 'day_sleep']
        main_sleep = sleeps_df[sleeps_df['sleep_type'].isin(main_sleep_types)].copy()

        print(f"  - 主要睡眠記錄: {len(main_sleep)} 筆")

        # 準備配對
        main_sleep['merge_date'] = main_sleep['sleep_date_final']
        dailies_df['merge_date'] = pd.to_datetime(dailies_df['date']).dt.date

        # 檢查並去重活動資料
        daily_duplicates = dailies_df.groupby(['_userId', 'merge_date']).size()
        daily_duplicates = daily_duplicates[daily_duplicates > 1]
        if len(daily_duplicates) > 0:
            print(f"  - 發現 {len(daily_duplicates)} 個活動資料重複，進行去重")
            dailies_df = dailies_df.drop_duplicates(subset=['_userId', 'merge_date'])

        # 執行配對
        merged_df = pd.merge(
            dailies_df,
            main_sleep[['_userId', 'merge_date', 'summaryId', 'sleep_hours',
                       'sleep_type', 'deepSleepDurationInSeconds',
                       'lightSleepDurationInSeconds', 'remSleepInSeconds']],
            on=['_userId', 'merge_date'],
            how='inner',
            suffixes=('_activity', '_sleep')
        )

        # 統計配對結果
        self.stats['配對成功數'] = len(merged_df)
        self.stats['配對率'] = len(merged_df) / len(main_sleep) * 100
        print(f"  - 成功配對: {len(merged_df)} 筆")
        print(f"  - 配對率: {self.stats['配對率']:.1f}%")
        print(f"  - 涵蓋使用者: {merged_df['_userId'].nunique()} 位")

        # 加入額外的睡眠品質指標
        merged_df['sleep_efficiency'] = merged_df.apply(
            lambda row: (row['deepSleepDurationInSeconds'] +
                        row['lightSleepDurationInSeconds'] +
                        row['remSleepInSeconds']) / (row['sleep_hours'] * 3600),
            axis=1
        )

        merged_df['deep_sleep_ratio'] = merged_df['deepSleepDurationInSeconds'] / (merged_df['sleep_hours'] * 3600)

        return merged_df

    def generate_cleaning_report(self):
        """生成清理報告"""
        print("\n" + "="*50)
        print("資料清理總結報告")
        print("="*50)

        print(f"\n1. 原始資料")
        print(f"   - 睡眠記錄: {self.stats.get('原始睡眠記錄', 0)} 筆")
        print(f"   - 活動記錄: {self.stats.get('原始活動記錄', 0)} 筆")

        print(f"\n2. 時區修正")
        print(f"   - 正常夜間睡眠: {self.stats.get('時區修正後正常睡眠', 0)} 筆")

        print(f"\n3. 異常值處理")
        print(f"   - 移除異常值: {self.stats.get('異常值移除數', 0)} 筆")

        print(f"\n4. 去重處理")
        print(f"   - 移除重複: {self.stats.get('去重移除數', 0)} 筆")

        print(f"\n5. 日期歸屬")
        print(f"   - 匹配率: {self.stats.get('日期匹配率', 0):.1f}%")

        print(f"\n6. 最終結果")
        print(f"   - 成功配對記錄: {self.stats.get('配對成功數', 0)} 筆")
        print(f"   - 配對率: {self.stats.get('配對率', 0):.1f}%")
        print(f"   - 資料保留率: {self.stats.get('配對成功數', 0) / self.stats.get('原始睡眠記錄', 1) * 100:.1f}%")

# 主執行函數
def run_complete_cleaning():
    """執行完整的資料清理"""
    # 設定檔案路徑
    sleeps_path = '/content/drive/MyDrive/多變量專案/sleeps.csv'
    dailies_path = '/content/drive/MyDrive/多變量專案/dailies.csv'

    # 建立清理器並執行
    cleaner = GarminDataCompleteCleaner()
    sleeps_cleaned, merged_data = cleaner.run_complete_cleaning(sleeps_path, dailies_path)

    # 儲存結果
    print("\n儲存清理後的資料...")
    sleeps_cleaned.to_csv('/content/drive/MyDrive/多變量專案/sleeps_fully_cleaned.csv', index=False)
    merged_data.to_csv('/content/drive/MyDrive/多變量專案/sleep_activity_merged_final.csv', index=False)

    # 儲存清理統計
    stats_df = pd.DataFrame([cleaner.stats])
    stats_df.to_csv('/content/drive/MyDrive/多變量專案/cleaning_statistics.csv', index=False)

    print("\n資料清理完成！")
    print(f"- 清理後睡眠資料: sleeps_fully_cleaned.csv")
    print(f"- 配對後整合資料: sleep_activity_merged_final.csv")
    print(f"- 清理統計報告: cleaning_statistics.csv")

    return sleeps_cleaned, merged_data

# 執行清理
if __name__ == "__main__":
    sleeps_final, merged_final = run_complete_cleaning()


> **資料處理限制**



儘管經過系統性的資料清理，日期歸屬邏輯仍存在挑戰：
- 原始日期與計算日期的匹配率為 61.5%
- 可能原因包括：
  1. Garmin 裝置的日期記錄邏輯不一致
  2. 跨時區旅行的特殊情況
  3. 裝置同步延遲

然而，最終 86.9% 的睡眠記錄成功配對活動資料，
確保了分析的有效性。未來研究可進一步探討優化
日期歸屬演算法。

In [ ]:
# 載入清理後的資料
final_data = pd.read_csv('/content/drive/MyDrive/多變量專案/sleep_activity_merged_final.csv')

# 基礎統計
print(final_data.describe())